In [ ]:
"""
ETHIOPIAN STUDENT DATA TRANSFORMER - PROFESSIONAL ETL PIPELINE
Fixed for column pattern: Grade_1_Math_Test_Score
"""

import pandas as pd
import numpy as np
import os
import logging

# =========================
# CONFIG
# =========================
INPUT_FILE = r"C:/Users/DELL/Documents/project_data/data/ethiopian_students_dataset.csv"
OUTPUT_FOLDER = r"C:/Users/DELL/Documents/project_data/powerbi_ready/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================
# LOGGING SETUP
# =========================
logging.basicConfig(
    filename=os.path.join(OUTPUT_FOLDER, "etl_log.txt"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print("=" * 70)
print("ETHIOPIAN STUDENT DATA TRANSFORMER - PROFESSIONAL VERSION")
print("=" * 70)

In [ ]:
# =========================
# LOAD DATA
# =========================
try:
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    logging.info("Data loaded successfully")
    print(f"\n✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
except Exception as e:
    logging.error(f"Data loading failed: {e}")
    raise

# =========================
# VALIDATION CHECK
# =========================
required_cols = ["Student_ID", "Overall_Average"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# =========================
# SAFE SAVE FUNCTION
# =========================
def save(df, name):
    if df is None or df.empty:
        logging.warning(f"{name} not saved (empty)")
        return
    path = os.path.join(OUTPUT_FOLDER, name)
    df.to_csv(path, index=False)
    logging.info(f"Saved {name} with {len(df)} rows")
    print(f"  ✓ {name} ({len(df):,} rows)")

# =========================
# FIXED PARSER - For Grade_1_Math_Test_Score pattern
# =========================
def parse_column(col):
    """Parse columns like: Grade_1_Math_Test_Score"""
    parts = col.split('_')
    
    if len(parts) < 4 or parts[0] != "Grade":
        return None, None, None
    
    grade = parts[1]  # "1"
    
    # Metric is the last part (or last two for compound metrics)
    if parts[-2] == 'Test' and parts[-1] == 'Score':
        metric = 'Test_Score'
        subject_parts = parts[2:-2]
    elif parts[-1] == 'Attendance':
        metric = 'Attendance'
        subject_parts = parts[2:-1]
    elif parts[-2] == 'Homework' and parts[-1] == 'Completion':
        metric = 'Homework_Completion'
        subject_parts = parts[2:-2]
    elif parts[-2] == 'Class' and parts[-1] == 'Participation':
        metric = 'Class_Participation'
        subject_parts = parts[2:-2]
    elif parts[-1] == 'Textbook':
        metric = 'Textbook'
        subject_parts = parts[2:-1]
    else:
        return None, None, None
    
    subject = "_".join(subject_parts).replace("_", " ")
    return grade, subject, metric